<a href="https://colab.research.google.com/github/Ayushkumarip/CG/blob/main/ass2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

In [3]:
!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz

In [4]:
os.environ["HADOOP_HOME"] = "/content/hadoop-3.3.6"
os.environ["PATH"] += ":/content/hadoop-3.3.6/bin"

In [5]:
!hadoop version

Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /content/hadoop-3.3.6/share/hadoop/common/hadoop-common-3.3.6.jar


In [6]:
!wget -q https://downloads.apache.org/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz

tar: spark-3.5.1-bin-hadoop3.tgz: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now


In [7]:
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["PATH"] += ":/content/spark-3.5.1-bin-hadoop3/bin"

In [8]:
!pip install pyspark

In [9]:
!wget -q https://downloads.apache.org/kafka/3.7.0/kafka_2.13-3.7.0.tgz
!tar -xzf kafka_2.13-3.7.0.tgz

tar (child): kafka_2.13-3.7.0.tgz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now


In [10]:
!/content/kafka_2.13-3.7.0/bin/zookeeper-server-start.sh -daemon /content/kafka_2.13-3.7.0/config/zookeeper.properties

/bin/bash: line 1: /content/kafka_2.13-3.7.0/bin/zookeeper-server-start.sh: No such file or directory


In [11]:
!/content/kafka_2.13-3.7.0/bin/kafka-server-start.sh -daemon /content/kafka_2.13-3.7.0/config/server.properties

/bin/bash: line 1: /content/kafka_2.13-3.7.0/bin/kafka-server-start.sh: No such file or directory


In [12]:
!/content/kafka_2.13-3.7.0/bin/kafka-topics.sh --create --topic twitter_stream --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1


/bin/bash: line 1: /content/kafka_2.13-3.7.0/bin/kafka-topics.sh: No such file or directory


In [14]:
!pip install kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.1/326.1 kB 8.8 MB/s eta 0:00:00


In [15]:
from kafka import KafkaProducer
import time
import json
import random

producer = KafkaProducer(bootstrap_servers='localhost:9092')

hashtags = ["#AI", "#BigData", "#Spark", "#Hadoop", "#ML"]

for i in range(100):
    tweet = {
        "text": "Learning " + random.choice(hashtags)
    }
    producer.send('twitter_stream', json.dumps(tweet).encode('utf-8'))
    time.sleep(1)

ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-producer-1, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: Connect attempt returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-producer-1, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-producer-1, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Connect attempt returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-producer-1, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-producer-1, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: Connect attempt retu

NoBrokersAvailable: NoBrokersAvailable

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession.builder \
    .appName("TwitterStreaming") \
    .getOrCreate()

df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "twitter_stream") \
    .load()

df = df.selectExpr("CAST(value AS STRING)")

hashtags = df.select(
    explode(split(df.value, " ")).alias("word")
).filter("word LIKE '#%'")

counts = hashtags.groupBy("word").count()

query = counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

In [16]:
!pkill -f kafka
!pkill -f zookeeper

In [17]:
!/content/kafka_2.13-3.7.0/bin/zookeeper-server-start.sh /content/kafka_2.13-3.7.0/config/zookeeper.properties > zookeeper.log 2>&1 &

In [18]:
!sleep 5

In [19]:
!ps aux | grep zookeeper

root       26124  0.0  0.0   7372  3600 ?        S    19:59   0:00 /bin/bash -c ps aux | grep zookeeper
root       26126  0.0  0.0   6480  2464 ?        S    19:59   0:00 grep zookeeper


In [20]:
!/content/kafka_2.13-3.7.0/bin/kafka-server-start.sh /content/kafka_2.13-3.7.0/config/server.properties > kafka.log 2>&1 &

In [21]:
!sleep 10

In [23]:
!netstat -tlnp | grep 9092

In [24]:
!/content/kafka_2.13-3.7.0/bin/kafka-topics.sh --create --topic twitter_stream --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1

/bin/bash: line 1: /content/kafka_2.13-3.7.0/bin/kafka-topics.sh: No such file or directory


In [25]:
!ls /content

hadoop-3.3.6  hadoop-3.3.6.tar.gz  kafka.log  sample_data  zookeeper.log


In [26]:
!tar -xzf kafka_2.13-3.7.0.tgz

tar (child): kafka_2.13-3.7.0.tgz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now


In [27]:
!pkill -f kafka
!pkill -f zookeeper

In [28]:
!/content/kafka_2.13-3.7.0/bin/zookeeper-server-start.sh /content/kafka_2.13-3.7.0/config/zookeeper.properties > zookeeper.log 2>&1 &
!sleep 5

In [29]:
!/content/kafka_2.13-3.7.0/bin/kafka-server-start.sh /content/kafka_2.13-3.7.0/config/server.properties > kafka.log 2>&1 &
!sleep 10

In [30]:
!netstat -tlnp | grep 9092

In [31]:
!/content/kafka_2.13-3.7.0/bin/kafka-topics.sh \
--create \
--topic twitter_stream \
--bootstrap-server 127.0.0.1:9092 \
--partitions 1 \
--replication-factor 1

/bin/bash: line 1: /content/kafka_2.13-3.7.0/bin/kafka-topics.sh: No such file or directory


In [32]:
!ls /content

hadoop-3.3.6  hadoop-3.3.6.tar.gz  kafka.log  sample_data  zookeeper.log


In [33]:
!wget https://downloads.apache.org/kafka/3.7.0/kafka_2.13-3.7.0.tgz

--2026-05-03 20:03:22--  https://downloads.apache.org/kafka/3.7.0/kafka_2.13-3.7.0.tgz
Resolving downloads.apache.org (downloads.apache.org)... 95.216.224.44, 88.99.208.237, 2a01:4f9:2b:1cc2::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|95.216.224.44|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-03 20:03:22 ERROR 404: Not Found.



In [34]:
!wget https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz

--2026-05-03 20:03:43--  https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 113466638 (108M) [application/x-gzip]
Saving to: ‘kafka_2.13-3.6.1.tgz’

kafka_2.13-3.6.1.tg 100%[===================>] 108.21M  11.5MB/s    in 11s     

2026-05-03 20:03:55 (9.92 MB/s) - ‘kafka_2.13-3.6.1.tgz’ saved [113466638/113466638]



In [35]:
!tar -xzf kafka_2.13-3.6.1.tgz

In [36]:
!ls /content | grep kafka

kafka_2.13-3.6.1
kafka_2.13-3.6.1.tgz
kafka.log


In [37]:
/content/kafka_2.13-3.6.1/bin/

SyntaxError: invalid syntax (2096369660.py, line 1)

In [38]:
!/content/kafka_2.13-3.6.1/bin/kafka-topics.sh --list --bootstrap-server 127.0.0.1:9092

[2026-05-03 20:04:48,483] WARN [AdminClient clientId=adminclient-1] Connection to node -1 (/127.0.0.1:9092) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-05-03 20:04:48,592] WARN [AdminClient clientId=adminclient-1] Connection to node -1 (/127.0.0.1:9092) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-05-03 20:04:48,694] WARN [AdminClient clientId=adminclient-1] Connection to node -1 (/127.0.0.1:9092) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-05-03 20:04:48,897] WARN [AdminClient clientId=adminclient-1] Connection to node -1 (/127.0.0.1:9092) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-05-03 20:04:49,401] WARN [AdminClient clientId=adminclient-1] Connection to node -1 (/127.0.0.1:9092) could not be established. Broker may not be available. (org.apache.kafka.cli

In [39]:
!tail -n 50 kafka.log

/bin/bash: line 1: /content/kafka_2.13-3.7.0/bin/kafka-server-start.sh: No such file or directory


In [40]:
!ls /content | grep kafka

kafka_2.13-3.6.1
kafka_2.13-3.6.1.tgz
kafka.log


In [41]:
!pkill -f kafka
!pkill -f zookeeper

In [42]:
!/content/kafka_2.13-3.6.1/bin/zookeeper-server-start.sh \
/content/kafka_2.13-3.6.1/config/zookeeper.properties > zookeeper.log 2>&1 &

In [43]:
!/content/kafka_2.13-3.6.1/bin/kafka-server-start.sh \
/content/kafka_2.13-3.6.1/config/server.properties > kafka.log 2>&1 &

In [44]:
!sleep 10

In [45]:
!netstat -tlnp | grep 9092

tcp6       0      0 :::9092                 :::*                    LISTEN      28619/java          


In [46]:
!/content/kafka_2.13-3.6.1/bin/kafka-topics.sh \
--create \
--topic twitter_stream \
--bootstrap-server 127.0.0.1:9092 \
--partitions 1 \
--replication-factor 1

Created topic twitter_stream.


In [47]:
!/content/kafka_2.13-3.6.1/bin/kafka-topics.sh \
--list \
--bootstrap-server 127.0.0.1:9092

twitter_stream


In [48]:
bootstrap_servers='127.0.0.1:9092'

In [49]:
from kafka import KafkaProducer
import json, time, random

producer = KafkaProducer(bootstrap_servers='127.0.0.1:9092')

hashtags = ["#AI", "#BigData", "#Spark", "#Hadoop", "#ML"]

for i in range(20):
    tweet = {"text": "Learning " + random.choice(hashtags)}
    producer.send('twitter_stream', json.dumps(tweet).encode('utf-8'))
    print("Sent:", tweet)
    time.sleep(1)

Sent: {'text': 'Learning #ML'}
Sent: {'text': 'Learning #Spark'}
Sent: {'text': 'Learning #BigData'}
Sent: {'text': 'Learning #Spark'}
Sent: {'text': 'Learning #Spark'}
Sent: {'text': 'Learning #ML'}
Sent: {'text': 'Learning #BigData'}
Sent: {'text': 'Learning #AI'}
Sent: {'text': 'Learning #Spark'}
Sent: {'text': 'Learning #BigData'}
Sent: {'text': 'Learning #Spark'}
Sent: {'text': 'Learning #AI'}
Sent: {'text': 'Learning #Hadoop'}
Sent: {'text': 'Learning #Hadoop'}
Sent: {'text': 'Learning #ML'}
Sent: {'text': 'Learning #Hadoop'}
Sent: {'text': 'Learning #ML'}
Sent: {'text': 'Learning #AI'}
Sent: {'text': 'Learning #ML'}
Sent: {'text': 'Learning #AI'}


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession.builder \
    .appName("TwitterStreaming") \
    .getOrCreate()

df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "127.0.0.1:9092") \
    .option("subscribe", "twitter_stream") \
    .load()

df = df.selectExpr("CAST(value AS STRING)")

hashtags = df.select(
    explode(split(df.value, " ")).alias("word")
).filter("word LIKE '#%'")

counts = hashtags.groupBy("word").count()

query = counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

In [51]:
!ls /content | grep spark

In [52]:
!wget https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar -xzf spark-3.5.1-bin-hadoop3.tgz

--2026-05-03 20:08:44--  https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400446614 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.1-bin-hadoop3.tgz’

spark-3.5.1-bin-had 100%[===================>] 381.90M  79.6MB/s    in 5.2s    

2026-05-03 20:08:49 (74.1 MB/s) - ‘spark-3.5.1-bin-hadoop3.tgz’ saved [400446614/400446614]



In [53]:
import os

os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["PATH"] += ":/content/spark-3.5.1-bin-hadoop3/bin"

In [54]:
!pip install pyspark

In [55]:
!ls /content/spark-3.5.1-bin-hadoop3/bin | grep spark-submit

spark-submit
spark-submit2.cmd
spark-submit.cmd


In [ ]:
import os
os.kill(os.getpid(), 9)

In [2]:
from pyspark.sql import SparkSession

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession.builder \
    .appName("TwitterStreaming") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.13:3.5.1") \
    .getOrCreate()

df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "127.0.0.1:9092") \
    .option("subscribe", "twitter_stream") \
    .load()

df = df.selectExpr("CAST(value AS STRING)")

hashtags = df.select(
    explode(split(df.value, " ")).alias("word")
).filter("word LIKE '#%'")

counts = hashtags.groupBy("word").count()

query = counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

Py4JJavaError: An error occurred while calling o59.load.
: java.lang.NoClassDefFoundError: Could not initialize class org.apache.spark.sql.kafka010.KafkaSourceProvider$
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.org$apache$spark$sql$kafka010$KafkaSourceProvider$$validateStreamOptions(KafkaSourceProvider.scala:338)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.sourceSchema(KafkaSourceProvider.scala:71)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceSchema(DataSource.scala:244)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo$lzycompute(DataSource.scala:129)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo(DataSource.scala:129)
	at org.apache.spark.sql.execution.streaming.StreamingRelation$.apply(StreamingRelation.scala:37)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:86)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$appl

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TwitterStreaming") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.13:3.5.1,"
            "org.apache.kafka:kafka-clients:3.6.1") \
    .getOrCreate()

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import socket, time, random, threading

def start_server():
    host = "0.0.0.0"
    port = 9999

    server = socket.socket()
    server.bind((host, port))
    server.listen(1)

    print("Server started... waiting for Spark")

    conn, addr = server.accept()
    print("Connected!")

    hashtags = ["#AI", "#BigData", "#Spark", "#Hadoop", "#ML"]

    while True:
        msg = "Learning " + random.choice(hashtags)
        conn.send((msg + "\n").encode())
        time.sleep(1)

threading.Thread(target=start_server).start()

Waiting for connection...


KeyboardInterrupt: 

In [ ]:
query = counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

In [2]:
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar -xzf spark-3.5.1-bin-hadoop3.tgz
!pip install pyspark

In [3]:
import os
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["PATH"] += ":/content/spark-3.5.1-bin-hadoop3/bin"

In [1]:
!fuser -k 9999/tcp

In [2]:
import socket, time, random, threading

def stream_data():
    host = "0.0.0.0"
    port = 9999

    server = socket.socket()
    server.bind((host, port))
    server.listen(1)

    print("📡 Server started... waiting for Spark")

    conn, addr = server.accept()
    print("✅ Spark connected!")

    hashtags = ["#AI", "#BigData", "#Spark", "#Hadoop", "#ML"]

    while True:
        msg = "Learning " + random.choice(hashtags)
        conn.send((msg + "\n").encode())
        print("Sent:", msg)
        time.sleep(1)

threading.Thread(target=stream_data).start()

📡 Server started... waiting for Spark


In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col
import time

spark = SparkSession.builder \
    .appName("SocialMediaStreaming") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Read stream
df = spark.readStream \
    .format("socket") \
    .option("host", "127.0.0.1") \
    .option("port", 9999) \
    .load()

# Extract hashtags
hashtags = df.select(
    explode(split(col("value"), " ")).alias("word")
).filter(col("word").startswith("#"))

# Count
counts = hashtags.groupBy("word").count()

# Write to memory (important fix)
query = counts.writeStream \
    .format("memory") \
    .queryName("hashtags_table") \
    .outputMode("complete") \
    .trigger(processingTime="2 seconds") \
    .start()

# 🔥 WAIT + CHECK LOOP (this is the key)
for i in range(5):
    time.sleep(5)
    print(f"\n=== Batch {i+1} ===")
    spark.sql("SELECT * FROM hashtags_table").show()

# Stop query
query.stop()

Sent: Learning #ML
Sent: Learning #Hadoop
Sent: Learning #ML
Sent: Learning #Spark
Sent: Learning #Spark
Sent: Learning #ML

=== Batch 1 ===
+----+-----+
|word|count|
+----+-----+
+----+-----+

Sent: Learning #AI
Sent: Learning #Hadoop
Sent: Learning #AI
Sent: Learning #AI
Sent: Learning #BigData

=== Batch 2 ===
+----+-----+
|word|count|
+----+-----+
+----+-----+

Sent: Learning #BigData
Sent: Learning #AI
Sent: Learning #ML
Sent: Learning #AI
Sent: Learning #ML

=== Batch 3 ===
+----+-----+
|word|count|
+----+-----+
+----+-----+

Sent: Learning #Spark
Sent: Learning #Spark
Sent: Learning #AI
Sent: Learning #BigData
Sent: Learning #ML

=== Batch 4 ===
+----+-----+
|word|count|
+----+-----+
+----+-----+

Sent: Learning #Spark
Sent: Learning #Spark
Sent: Learning #ML
Sent: Learning #Spark
Sent: Learning #AI

=== Batch 5 ===
Sent: Learning #Hadoop
+----+-----+
|word|count|
+----+-----+
+----+-----+

Sent: Learning #Spark
Sent: Learning #BigData
Sent: Learning #Hadoop
Sent: Learning #Hado

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [12]:
df.writeStream.format("memory").queryName("raw").start()

time.sleep(10)
spark.sql("SELECT * FROM raw").show()

Sent: Learning #AI
Sent: Learning #BigData
Sent: Learning #ML
Sent: Learning #AI
Sent: Learning #ML
Sent: Learning #Spark
Sent: Learning #ML
Sent: Learning #Hadoop
Sent: Learning #BigData
Sent: Learning #Spark
Sent: Learning #BigData
+-----+
|value|
+-----+
+-----+



In [9]:
counts.writeStream \
    .format("memory") \
    .queryName("table") \
    .outputMode("complete") \
    .start()

import time
time.sleep(10)

spark.sql("SELECT * FROM table").show()

Sent: Learning #ML
Sent: Learning #Hadoop
Sent: Learning #Hadoop
Sent: Learning #Spark
Sent: Learning #ML
Sent: Learning #ML
Sent: Learning #Spark
Sent: Learning #AI
Sent: Learning #ML
Sent: Learning #ML
Sent: Learning #Hadoop
+----+-----+
|word|count|
+----+-----+
+----+-----+

